# 05 — Divergence Judge

**Owner:** Ian Schmitt (T3) · **Reads:** `outputs/predictions/*.parquet` (from 02/03/04), `data/processed/splits.parquet`, `data/golden/golden_set.csv` · **Writes:** `outputs/tables/05-judge_*.csv`, `outputs/figures/05-judge_*.png`

Where the two classifiers disagree, a locally served LLM adjudicates each review **blind** — no gold label, no model votes — as an independent third opinion. This notebook carries the judge machinery, its frozen configuration, the golden-set evaluation that froze it, and (once 02/03 land prediction files) the disagreement adjudication and taxonomy analysis.

Development record: [`../docs/judge-dev-log.md`](../docs/judge-dev-log.md). Annotation guideline: [`../docs/tagging-protocol.md`](../docs/tagging-protocol.md).

**Status: scaffold.** The judge configuration and golden-set evaluation are live and reproducible; the disagreement sections activate when the prediction files exist and report what is missing until then, so the notebook always runs clean top to bottom.

In [1]:
# Shared foundation from src/shared.py (landed with 00_core).
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from shared import PATHS

In [2]:
import json
import time

import pandas as pd
from dotenv import dotenv_values
from openai import OpenAI

TBL = PATHS["tables_dir"]
PREDICTIONS = PATHS["predictions_dir"]
GOLDEN_CSV = PATHS["repo_root"] / "data" / "golden" / "golden_set.csv"


## Frozen judge configuration

Ratified 2026-07-23 after the golden-set evaluations (full history in the dev log): **prompt v3**, temperature 0, reasoning disabled, output schema-constrained to a forced binary choice, blind. The endpoint is provider-agnostic: `LLM_BASE_URL` / `LLM_API_KEY` / `LLM_MODEL` come from the gitignored `.env` (`cp .env.example .env`), so the same code targets a local llama.cpp server or any OpenAI-compatible cloud endpoint. Temperature and the no-thinking flag ride in **every request**, so the frozen behavior holds no matter how the serving side was launched.

In [3]:
env = dotenv_values(PATHS["repo_root"] / ".env")
if not env.get("LLM_BASE_URL"):
    raise RuntimeError("No LLM endpoint configured: cp .env.example .env and fill it in.")

client = OpenAI(
    base_url=env["LLM_BASE_URL"], api_key=env["LLM_API_KEY"], timeout=600, max_retries=2
)
MODEL = env["LLM_MODEL"]

JUDGE_PROMPT = (
    "You are a sentiment adjudicator for IMDB movie reviews. Each review's author "
    "also gave the film a star rating from 1 to 10, and the dataset's label is "
    "derived from that rating (>= 7 positive, <= 4 negative). Read the review, "
    "infer the reviewer's verdict, and classify it as positive or negative.\n\n"
    "Judge the reviewer's verdict on the film, not the quality of their writing "
    "and not your own opinion of the film. Where a review is mixed, decide which "
    "way the weighing lands. Where praise or criticism is ironic, the intended "
    "meaning governs, not the surface words. You must choose exactly one label, "
    "even for mixed reviews. Respond in JSON."
)

JUDGE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "verdict",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "label": {"type": "string", "enum": ["positive", "negative"]}
            },
            "required": ["label"],
            "additionalProperties": False,
        },
    },
}


def judge(text):
    """One blind adjudication: review text in, 'positive' or 'negative' out.

    temperature=0 and enable_thinking=false are set per request so the frozen
    config holds on any endpoint; the schema constraint makes malformed output
    impossible by construction.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": text},
        ],
        response_format=JUDGE_SCHEMA,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return json.loads(resp.choices[0].message.content)["label"]

## Golden-set evaluation

The instrument that selected and froze the configuration: 32 hand-curated IMDB **train**-split reviews (17 negative / 14 positive; 10 easy / 22 hard across negation, mixed, sarcasm, and short/noisy), sourced from the train split so prompt development never touches data that could reach the real adjudication runs. Frozen result: **31/32 overall (96.9%), 21/22 hard (95.5%)**. The single miss, [17596], is the label-noise exemplar — its prose argues against its own gold label — kept in the set deliberately. Re-running this section reproduces the numbers (greedy decoding is deterministic for a fixed server build); with the checkpoint file present it costs zero API calls.

In [4]:
def adjudicate(frame, out_csv, id_col="id", text_col="text"):
    """Run the judge over a dataframe, checkpointing one row per verdict.

    Interrupt-safe: ids already present in out_csv are skipped on re-run, so a
    killed run resumes where it stopped and a completed run costs nothing.
    """
    out_csv = pathlib.Path(out_csv)
    if out_csv.exists():
        done = pd.read_csv(out_csv)
    else:
        done = pd.DataFrame(columns=[id_col, "judge_label", "latency_s"])
    todo = frame[~frame[id_col].isin(done[id_col])]
    print(f"{len(done)} checkpointed, {len(todo)} to run -> {out_csv.name}")
    for row in todo.itertuples():
        t0 = time.perf_counter()
        label = judge(getattr(row, text_col))
        verdict = {
            id_col: getattr(row, id_col),
            "judge_label": label,
            "latency_s": round(time.perf_counter() - t0, 2),
        }
        done = pd.concat([done, pd.DataFrame([verdict])], ignore_index=True)
        done.to_csv(out_csv, index=False)
    return done

In [5]:
golden = pd.read_csv(GOLDEN_CSV)
verdicts = adjudicate(
    golden, TBL / "05-judge_golden_eval.csv", id_col="train_idx", text_col="text"
)

ev = golden.merge(verdicts, on="train_idx")
ev["correct"] = ev["judge_label"] == ev["gold"]
hard = ev[ev["difficulty"] == "hard"]
print(
    f"overall {ev['correct'].sum()}/{len(ev)} = {ev['correct'].mean():.1%} | "
    f"hard {hard['correct'].sum()}/{len(hard)} = {hard['correct'].mean():.1%}"
)
for miss in ev[~ev["correct"]].itertuples():
    print(f"miss [{miss.train_idx}] {miss.category}: gold={miss.gold}")

32 checkpointed, 0 to run -> 05-judge_golden_eval.csv
overall 31/32 = 96.9% | hard 21/22 = 95.5%
miss [17596] mixed: gold=positive


## Disagreement adjudication — waiting on 02/03

A disagreement is `y_pred_lr != y_pred_nn`, computed in one merge from the committed prediction files with review text rehydrated from `splits.parquet` (the tagging protocol's Inputs section specifies the blinding). The guard below activates the remaining sections when the prediction files exist.

In [6]:
NEEDED = ["lr_val.parquet", "nn_val.parquet"]
missing = [name for name in NEEDED if not (PREDICTIONS / name).exists()]
HAVE_PREDICTIONS = not missing
if HAVE_PREDICTIONS:
    print("prediction files present - disagreement sections active")
else:
    print("waiting on:", ", ".join(missing), "(02/03 not merged yet)")

waiting on: lr_val.parquet, nn_val.parquet (02/03 not merged yet)


In [7]:
def judge_vs_baseline(merged, id_col="id"):
    """Score the judge against the pre-registered baseline on a disagreement set.

    Where the two models disagree, exactly one of them is correct on every row,
    so acc_lr + acc_nn == 1 over the set. The null is therefore not a coin flip
    but "always side with the better model", i.e. max(acc_lr, acc_nn). The judge
    earns its place only by beating that.

    Registered in docs/judge-dev-log.md (2026-07-24) and committed before any
    prediction file existed, so the bar cannot be chosen after seeing the data.

    `merged` carries y_true, y_pred_lr, y_pred_nn and judge_label per id.
    """
    truth = merged["y_true"]
    acc_lr = float((merged["y_pred_lr"] == truth).mean())
    acc_nn = float((merged["y_pred_nn"] == truth).mean())
    judged = merged["judge_label"].map({"negative": 0, "positive": 1})
    acc_judge = float((judged == truth).mean())
    baseline = max(acc_lr, acc_nn)
    return {
        "n": len(merged),
        "acc_lr": acc_lr,
        "acc_nn": acc_nn,
        "baseline_best_model": baseline,
        "acc_judge": acc_judge,
        "beats_baseline": acc_judge > baseline,
    }


# Sanity-check the arithmetic on a synthetic frame, so the function is exercised
# even while the real prediction files are still missing.
_probe = pd.DataFrame(
    {
        "y_true": [1, 1, 0, 0],
        "y_pred_lr": [1, 0, 0, 1],
        "y_pred_nn": [0, 1, 1, 0],
        "judge_label": ["positive", "positive", "negative", "negative"],
    }
)
_check = judge_vs_baseline(_probe)
assert _check["acc_lr"] + _check["acc_nn"] == 1.0, "disagreement identity violated"
print("baseline scorer ready:", _check)

baseline scorer ready: {'n': 4, 'acc_lr': 0.5, 'acc_nn': 0.5, 'baseline_best_model': 0.5, 'acc_judge': 1.0, 'beats_baseline': True}


### Planned once predictions land

1. **Derive the val disagreement set** (one merge) and apply the protocol's sampling rule: all if ≤50, else a seeded random 50.
2. **Screen for golden-set overlap.** The golden set is train-sourced and `val` is drawn from the same pool, so up to 8 golden reviews can reappear in a val disagreement set (`source_index` in `splits.parquet` makes this a one-line check). Report the count and exclude if nonzero. The test run needs no such screen: **zero** golden reviews have a text twin in the test split, verified 2026-07-24.
3. **Adjudicate** with `adjudicate()` — the same checkpointed runner exercised above.
4. **Score against the pre-registered baseline** using `judge_vs_baseline()` above: the judge must beat `max(acc_lr, acc_nn)`, not 50%. Reported either way.
5. **Tag** the sampled disagreements blind per the protocol; votes, gold, and judge verdicts rejoin only after tags are recorded.
6. **Analyze:** taxonomy distribution, per-category LR/NN win rates, judge accuracy by category, and the who-was-right readout; repeated on test disagreements after the single final run.

In [8]:
# Handoff manifest: every artifact this notebook wrote, for the record.
written = sorted(
    p.relative_to(PATHS["repo_root"]) for p in TBL.glob("05-judge_*.csv")
)
print(f"{len(written)} artifacts written:")
for p in written:
    print(" ", p)

1 artifacts written:
  outputs/tables/05-judge_golden_eval.csv
